# Module 3 · Generative AI & LLMs — Live GPU Demo
**Prof. Dr. R. Bhaduri · DSU_NVIDIA AI-First Core Team**

Four demos, each mapped to the Module 3 slides. Run them top to bottom.

| Cell | Module 3 concept | Slides |
|------|------------------|--------|
| 1 | Word embeddings — distance encodes meaning | 12–13 |
| 2 | Next-word prediction (context → the pangram) | 15–21 |
| 3 | Self-attention inside the Transformer | 18–19 |
| 4 | One foundation model, many tasks | 7, 9, 22 |

> Tested on an NVIDIA L40S Brev instance. Package versions are pinned in `setup.sh`
> (transformers 4.44.2 + torch 2.4.1/cu121) so a fresh instance runs without edits.


## 0 · Verify the GPU
Confirms we are really running on an NVIDIA GPU (the 'GPU in the cloud' Brev gives us).

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", device)

## 1 · Words as Vectors — *distance encodes meaning*  (Slides 12–13)
A **word embedding** turns each word into a vector so that *geometry* captures meaning.
GloVe stores words in **lowercase**, so we query in lowercase. Two slide claims, live:
related words sit close, and country→capital pairs share the same vector **direction**.

In [ ]:
import gensim.downloader as api
print("Loading GloVe embeddings (cached on the instance)...")
wv = api.load("glove-wiki-gigaword-100")   # 100-d vectors; vocabulary is lowercase

# (a) Related words cluster together  — slide 12
for a, b in [("compute", "calculate"), ("microwave", "mixer"), ("king", "cat")]:
    print(f"similarity({a:10s},{b:10s}) = {wv.similarity(a, b):.3f}")

print()
# (b) Analogy = geometry (lowercase words)  — slide 13 is the capital example
print("king   - man     + woman   ->", wv.most_similar(positive=['king', 'woman'],   negative=['man'])[0])
print("paris  - france  + germany ->", wv.most_similar(positive=['paris', 'germany'], negative=['france'])[0])
print("london - england + japan   ->", wv.most_similar(positive=['london', 'japan'],  negative=['england'])[0])
print("\nParallel arrows on the slide = the same 'is-capital-of' direction in vector space.")

## 2 · Next-Word Prediction — *context changes the guess*  (Slides 15–21)
With **no context** the model hedges with punctuation/filler; with **full context** it knows the
classic pangram. The last line lets GPT-2 actually finish the sentence — the headline result.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

def top_next_words(prompt, k=5):
    ids = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = gpt2(**ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return [(tok.decode(i).strip(), round(float(p), 3)) for i, p in zip(top.indices, top.values)]

print("Weak context  'lazy'                          ->", top_next_words("lazy"))
print("Full context  'The quick brown fox ... lazy'  ->",
      top_next_words("The quick brown fox jumps over the lazy"))

# Headline: let the model finish the sentence -> '... lazy dog.'
ids = tok("The quick brown fox jumps over the lazy", return_tensors="pt").to(device)
with torch.no_grad():
    out = gpt2.generate(**ids, max_new_tokens=3, pad_token_id=tok.eos_token_id)
print("\nModel completes ->", tok.decode(out[0], skip_special_tokens=True))

## 3 · Self-Attention — *the Transformer's trick*  (Slides 18–19)
The heat-map shows, for each word, **which words it attends to** — the Multi-Head Attention
block made visible. We use a static matplotlib heat-map: it always renders in JupyterLab
(bertviz needs extra JavaScript) and reads clearly on a projector.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch, matplotlib.pyplot as plt

m_name = "gpt2"
atok = AutoTokenizer.from_pretrained(m_name)
amodel = AutoModel.from_pretrained(m_name, output_attentions=True,
                                   attn_implementation="eager").eval()

sentence = "The quick brown fox jumps over the lazy dog"
inputs = atok(sentence, return_tensors="pt")
with torch.no_grad():
    attn = amodel(**inputs).attentions           # one tensor per layer
tokens = [t.replace("\u0120", "") for t in atok.convert_ids_to_tokens(inputs["input_ids"][0])]

a = attn[-1][0].mean(0).detach().cpu().numpy()   # last layer, averaged over heads

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(a, cmap="Greens")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_xlabel("attended TO"); ax.set_ylabel("attending FROM")
ax.set_title("Self-attention — last layer, averaged over heads")
plt.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

In [ ]:
# --- OPTIONAL: interactive bertviz (needs require.js). Run this cell FIRST if you want it. ---
# from IPython.display import display, HTML
# display(HTML('<script src="https://cdnjs.cloudflare.com/ajax/libs/require.js/2.3.6/require.min.js"></script>'))
# from bertviz import head_view
# head_view(attn, tokens)

## 4 · One Foundation Model, Many Tasks  (Slides 7, 9, 22)
**One** model summarises, translates and answers. FLAN-T5 runs locally (no API key).
Note: the *base* model is small, so hard facts can miss — a nice lead-in to "bigger = better".
Uncomment the `large` lines for stronger Q&A (≈3 GB download; the L40S has room).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_id = "google/flan-t5-base"        # swap to "google/flan-t5-large" for stronger answers
t5tok = AutoTokenizer.from_pretrained(model_id)
t5 = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device).eval()

def ask(prompt, max_new_tokens=60):
    ids = t5tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = t5.generate(**ids, max_new_tokens=max_new_tokens)
    return t5tok.decode(out[0], skip_special_tokens=True)

print("SUMMARISE :", ask("Summarize: A foundation model is trained once on broad, "
                         "largely unlabelled data, then adapted to many downstream tasks."))
print("TRANSLATE :", ask("Translate to German: Generative AI creates new content from a prompt."))
print("Q & A     :", ask("Question: Which neural network architecture, introduced in 2017, "
                         "is based on self-attention? Answer with its name."))

## Recap
You have now *seen* every box from the module: words → vectors, context → prediction,
attention → the Transformer, and one foundation model doing many tasks.
**Stop this instance in the Brev tab when finished** to save GPU credits.